# Evo2 DNA Sequence Generation and Scoring

This notebook demonstrates examples of how to use both tools involving the Evo2 model. In the first section, we use `run_evo2_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling with KV caching for efficient generation. In the second section, we use `run_evo2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model.

In [14]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo2")
display_overview("evo2")
display_docs_section("evo2", "Background")

# Evo2

> Evo2 is Arc Institute's genome-scale DNA language model for sequence generation and scoring. Trained on billions of nucleotides spanning prokaryotic and eukaryotic genomes, Evo2 performs autoregressive generation of DNA from prompts and scores sequences by log-likelihood. The tool supports local GPU/CPU execution with KV caching for efficient long generations.

## Background

DNA encodes the instructions for all cellular processes. Beyond protein-coding genes (\~1.5% of the human genome), the vast majority of genomic sequence consists of regulatory elements, structural features, repetitive elements, and sequences of unknown function. The "grammar" of DNA -- the patterns that distinguish functional from non-functional sequence -- is complex and context-dependent.

Genome-scale language models like Evo2 learn these patterns by training on diverse genomic sequences across all domains of life. By predicting each nucleotide given preceding context, the model captures:

* **Codon usage patterns** and open reading frame structure in coding regions
* **Regulatory motifs** ([transcription factor](https://en.wikipedia.org/wiki/Transcription_factor) binding sites, [promoter](https://en.wikipedia.org/wiki/Promoter_\(genetics\)) elements, splice signals)
* **Compositional biases** ([GC content](https://en.wikipedia.org/wiki/GC-content), [CpG islands](https://en.wikipedia.org/wiki/CpG_island), repetitive elements)
* **Long-range dependencies** ([enhancer](https://en.wikipedia.org/wiki/Enhancer_\(genetics\))-promoter interactions, chromatin domain boundaries)
* **Cross-kingdom sequence grammar** (from viral genomes to mammalian chromosomes)

Evo2 uses byte-level tokenization (vocab\_size=512), where each DNA base maps to its ASCII value (A=65, C=67, G=71, T=84, N=78). This allows the model to handle any genomic sequence without a specialized tokenizer.

## Available tools

In [15]:
display_available_tools("evo2")

- **`run_evo2_sample()`** — Sample DNA sequences using Evo2 language model
- **`run_evo2_score()`** — Score DNA sequences using Evo2 language model

### `run_evo2_sample`

This tool utilizes Evo2 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over genomic DNA. The `temperature` and `top_k` parameters control sampling diversity: lower values produce conservative, high-confidence extensions while higher values allow more exploratory generation.

In [16]:
from proto_tools import (
    Evo2SampleInput,
    Evo2SampleConfig,
    run_evo2_sample,
)

In [17]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_sample")

# Create an Evo2 input with one DNA starting prompt
inputs = Evo2SampleInput(prompts=["ATGCGTAAA"])

**Input** — `Evo2SampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `List[string]` | required | Prompt sequences for DNA generation. Can be provided as: |

In [18]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_sample")

# Configure sampling parameters | see docs above for all fields
config = Evo2SampleConfig(
    num_tokens=200,
    temperature=0.8,
    top_k=4,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prepend_prompt` | `boolean` | `True` | Whether to prepend the input prompt to the generated sequence in the output. If `True`, the returned sequences include both the prompt and generated tokens. If `False`, only the newly generated tokens are returned. Default: `True`. |
| `model_checkpoint` | `enum` | `evo2_7b` | Evo2 model checkpoint to use. Options: Available options: `evo2_7b`, `evo2_20b`, `evo2_40b`, `evo2_7b_base`, `evo2_40b_base`, `evo2_1b_base`, `evo2_7b_262k`, `evo2_7b_microviridae` |
| `top_k` | `integer` | `4` | Limits sampling to the top-k most probable tokens at each generation step. Lower values make generation more focused and deterministic, higher values increase diversity. Must be at least 1. Default: 4. |
| `top_p` | `number` | `1` | Nucleus sampling parameter. Chooses the smallest set of tokens whose cumulative probability mass is at least `top_p`. Common values: |
| `temperature` | `number` | `1.0` | Scales the randomness of sampling by adjusting the sharpness of the probability distribution: |
| `num_tokens` | `integer` | `32` | Number of new tokens to generate per sequence (does not include the prompt tokens). For DNA sequences, each token represents a nucleotide or short subsequence depending on the tokenizer. Must be at least 1. Default: 32. |
| `cached_generation` | `boolean` | `True` | Whether to use vortex KV caching for faster generation. KV caching stores intermediate attention states to avoid recomputation. Recommended for long sequences. Default: `True`. |
| `max_seqlen` | `integer` |  | Optional maximum sequence length to generate. Determines the maximum size of the KV cache. If `None`, automatically calculated as `prompt_length + num_tokens`. Useful for constraining memory usage. Default: `None`. |
| `stop_at_eos` | `boolean` | `True` | Whether to stop generation when an end-of-sequence (EOS) token is encountered. If `True`, generation terminates early for sequences that naturally end. If `False`, always generates exactly `num_tokens` tokens. Default: `True`. |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. Default: `1`. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading from Hugging Face. Useful for offline inference or custom model versions. |
| `force_prompt_threshold` | `integer` |  | Optional number of tokens to prefill in parallel before switching to autoregressive prompt forcing. This can speed up generation for long prompts. Default: `None` (automatic determination). |
| `print_generation` | `boolean` | `True` | Whether to print generated tokens to console in real-time as they are generated. Useful for debugging and monitoring progress. Default: `True`. |
| `old_kv_cache` | `Dict[string, any]` |  | Dictionary of inference parameters containing a pre-computed KV cache from a previous generation run. Used for continuing generation from a cached state. Default: `None`. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run sampling on (`"cuda"`, `"cpu"`, `"mps"`). |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [19]:
# Run the sampling function
sample_result = run_evo2_sample(inputs, config)

Running evo2


In [20]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")

**Output** — `Evo2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Generated DNA sequences. Each sequence is a string of nucleotides. If `prepend_prompt=True` in the config, sequences include both the input prompt and newly generated tokens. If `False`, only the newly generated tokens are returned. |
| `logits` | `array` |  | Per-token logits for each generated sequence. |
| `kv_caches` | `array` |  | List of KV cache dictionaries, one per sequence. Each cache contains the intermediate attention states from generation. Can be passed as `old_kv_cache` in a subsequent generation call to continue generation from the cached state. Useful for: |

Prompt:       ATGCGTAAA
Generated:    ATGCGTAAACGTCATTATACGAAATATTATCAGCCAAAATATCATCCCCCAAAATATTCACGAACTCTTCCCAAAACAAC...
Total length: 209 nucleotides


### `run_evo2_score`

Evo2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each nucleotide is given its preceding context. The resulting perplexity provides a single-number summary of sequence naturalness: lower perplexity means the sequence more closely matches the statistical patterns in the training data. This is useful for ranking generated candidates, comparing wild-type and engineered variants, or filtering out sequences that deviate substantially from natural genomic patterns.

In [21]:
from proto_tools import (
    Evo2ScoringInput,
    Evo2ScoringConfig,
    run_evo2_score,
)

In [22]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_score")

# Score two short DNA sequences
score_inputs = Evo2ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

**Input** — `Evo2ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | DNA sequences to score. |

In [23]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_score")

# Configure scoring | see docs above for all fields
score_config = Evo2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `evo2_7b` | Evo2 model checkpoint to use. Currently available: `"evo2_7b"` (7B parameters), `"evo2_40b"` (40B parameters), and base/specialized variants. Default: `"evo2_7b"`. Available options: `evo2_7b`, `evo2_20b`, `evo2_40b`, `evo2_7b_base`, `evo2_40b_base`, `evo2_1b_base`, `evo2_7b_262k`, `evo2_7b_microviridae` |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. Default: `1`. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"`, `"cpu"`, or specific GPU devices like `"cuda:0"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [24]:
# Run the scoring function
score_result = run_evo2_score(score_inputs, score_config)

Running evo2


In [25]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[SequenceScores]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |
| `per_position_metrics` | `Dict[string, List[number]]` |  | Optional per-position scoring metrics, keyed by metric name. Each value is a list of length equal to the input sequence, with `None` at positions where that metric is not available. |

Sequence 1: ATGCGTAAA
    Log-likelihood:     -10.975
    Avg log-likelihood: -1.372
    Perplexity:         3.943
Sequence 2: ATGCGTAAATTT
    Log-likelihood:     -14.729
    Avg log-likelihood: -1.339
    Perplexity:         3.815


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [26]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo2_sequences.fasta'}")

score_result.export(name="evo2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo2_scores.csv'}")

Generated sequences exported to example_output/evo2_sequences.fasta
Scores exported to example_output/evo2_scores.csv
